Import packages
I use psycopg2 to connect Python with PostgreSQL.
I use pandas to put the database results in a dataframe.

In [2]:
import psycopg2
import pandas

Connect to PostgreSQL
I use this function to connect Python with my PostgreSQL database.
I need this connection before I can get data from the database.

In [3]:
def connect_db():
    connection = psycopg2.connect(
        host="localhost",
        database="movie_project",
        user="postgres",
        password="0000"
    )

    return connection

In [7]:
def connect_db():
    connection = psycopg2.connect(
        host="localhost",
        port="5433",
        database="movie_project",
        user="postgres",
        password="0000"
    )

    return connection

In [8]:
connection = connect_db()
print("Connection successful")
connection.close()

Connection successful


Function 1 - Expert reviews and worldwide box office
This function gets expert review data for each movie.
It supports the research question:
To what extent are expert and user evaluations associated with worldwide box-office performance?
I use average expert score, number of reviews, review length, positive emotion and negative emotion.
These variables were selected because they are supported by the literature in our report.

In [10]:
def get_expert_review_data():
    connection = connect_db()
    cursor = connection.cursor()

    cursor.execute("""
        SELECT
            movie.title,
            box_office.worldwide_box_office,
            AVG(expert_review.idvscore),
            COUNT(expert_review.expert_review_id),
            AVG(expert_review.liwc_wc),
            AVG(expert_review.liwc_posemo),
            AVG(expert_review.liwc_negemo)
        FROM movie
        JOIN expert_review
            ON movie.url = expert_review.url
        JOIN box_office
            ON movie.url = box_office.movie_url
        WHERE box_office.worldwide_box_office IS NOT NULL
        GROUP BY
            movie.url,
            movie.title,
            box_office.worldwide_box_office
        ORDER BY movie.title;
    """)

    rows = cursor.fetchall()

    columns = [
        "title",
        "worldwide_box_office",
        "average_expert_score",
        "expert_review_count",
        "average_word_count",
        "average_positive_emotion",
        "average_negative_emotion"
    ]

    dataframe = pandas.DataFrame(rows, columns=columns)

    cursor.close()
    connection.close()

    return dataframe

In [11]:
expert_data = get_expert_review_data()
expert_data.head()

,title,worldwide_box_office,average_expert_score,expert_review_count,average_word_count,average_positive_emotion,average_negative_emotion
0,'tis autumn: the search for jackie paris,4043,75.0000000000000000,7,25.0000000000000000,5.188571,0.000000
1,[rec],23266821,74.5555555555555556,9,24.8888888888888889,0.902222,4.287778
2,[rec] 2,18527766,54.0000000000000000,13,30.0000000000000000,1.449231,4.276923
3,[rec] 3: genesis,7250600,49.6363636363636364,11,30.5454545454545455,1.632727,3.480000
4,10 cloverfield lane,108286422,75.7441860465116279,43,31.6511627906976744,5.880233,3.393256


Function 2 - User reviews and worldwide box office
This function gets user review data for each movie.
It supports the same research question about evaluations and worldwide box office.
I use average user score, number of reviews, review length, positive emotion, negative emotion and helpfulness.
Helpfulness is only available for the user reviews.

In [13]:
def get_user_review_data():
    connection = connect_db()
    cursor = connection.cursor()

    cursor.execute("""
        SELECT
            movie.title,
            box_office.worldwide_box_office,
            AVG(user_review.idvscore),
            COUNT(user_review.consumer_review_id),
            AVG(user_review.liwc_wc),
            AVG(user_review.liwc_posemo),
            AVG(user_review.liwc_negemo),
            SUM(user_review.thumbsup)::FLOAT /
                NULLIF(SUM(user_review.thumbstot), 0)
        FROM movie
        JOIN user_review
            ON movie.url = user_review.url
        JOIN box_office
            ON movie.url = box_office.movie_url
        WHERE box_office.worldwide_box_office IS NOT NULL
        GROUP BY
            movie.url,
            movie.title,
            box_office.worldwide_box_office
        ORDER BY movie.title;
    """)

    rows = cursor.fetchall()

    columns = [
        "title",
        "worldwide_box_office",
        "average_user_score",
        "user_review_count",
        "average_word_count",
        "average_positive_emotion",
        "average_negative_emotion",
        "helpfulness_ratio"
    ]

    dataframe = pandas.DataFrame(rows, columns=columns)

    cursor.close()
    connection.close()

    return dataframe

In [14]:
user_data = get_user_review_data()
user_data.head()

,title,worldwide_box_office,average_user_score,user_review_count,average_word_count,average_positive_emotion,average_negative_emotion,helpfulness_ratio
0,'tis autumn: the search for jackie paris,4043,None,1,0E-20,0.000000,0.000000,NaN
1,[rec],23266821,6.0000000000000000,7,49.8571428571428571,1.442857,1.420000,0.400000
2,[rec] 2,18527766,6.9565217391304348,23,94.9565217391304348,4.172174,3.326957,0.538462
3,[rec] 3: genesis,7250600,4.6206896551724138,29,109.4827586206896552,3.133103,4.424138,0.457143
4,10 cloverfield lane,108286422,7.3034825870646766,202,157.1782178217821782,5.706733,2.360347,0.421622


Output
The two functions return analysis-ready dataframes.
These dataframes can later be used for statistical analysis of the research question.